In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/project_3/raw_data.csv')

In [ ]:
df.head()

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


In [ ]:
#remove stray quotes
df['Booking ID'] = df['Booking ID'].str.replace('"', '', regex=False).str.strip()
df['Customer ID'] = df['Customer ID'].str.replace('"', '', regex=False).str.strip()

In [ ]:
#NaN means "did not occur" -> 0, not missing
flag_cols = ['Cancelled Rides by Customer', 'Cancelled Rides by Driver', 'Incomplete Rides']
for col in flag_cols:
    df[col] = df[col].fillna(0).astype(int)

In [ ]:
#Combining Date + Time into one datetime for time-based analysis
df['Datetime'] = pd.to_datetime(df['Date'].astype(str) + ' ' + df['Time'].astype(str))
df['Hour'] = df['Datetime'].dt.hour
df['DayOfWeek'] = df['Datetime'].dt.day_name()

In [ ]:
df['Booking ID'].duplicated().sum()

np.int64(1233)

1233 duplicate Booking IDs

In [ ]:
dupes = df[df['Booking ID'].duplicated(keep=False)].sort_values('Booking ID')
exact_dupe_rows = df.duplicated().sum()  # fully identical rows across ALL columns
print(f"Fully identical duplicate rows: {exact_dupe_rows}")
print(f"Same Booking ID but different data: {1233 - exact_dupe_rows if exact_dupe_rows <= 1233 else 'check manually'}")

Fully identical duplicate rows: 0
Same Booking ID but different data: 1233


Duplicate Booking IDs (1,233, 0 fully identical rows): This means the same ID is attached to different bookings , a data generation defect, not duplicate transactions.

Booking ID is NOT unique, so we are going to use row index / row count as the booking grain, not Booking ID, for counts

In [ ]:
df['Row_ID'] = df.index

In [ ]:
df.columns

Index(['Date', 'Time', 'Booking ID', 'Booking Status', 'Customer ID',
       'Vehicle Type', 'Pickup Location', 'Drop Location', 'Avg VTAT',
       'Avg CTAT', 'Cancelled Rides by Customer',
       'Reason for cancelling by Customer', 'Cancelled Rides by Driver',
       'Driver Cancellation Reason', 'Incomplete Rides',
       'Incomplete Rides Reason', 'Booking Value', 'Ride Distance',
       'Driver Ratings', 'Customer Rating', 'Payment Method', 'Datetime',
       'Hour', 'DayOfWeek', 'Row_ID'],
      dtype='object')

In [ ]:
df = df[['Row_ID'] + [col for col in df.columns if col != 'Row_ID']]

In [ ]:
df.head()

,Row_ID,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,...,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method,Datetime,Hour,DayOfWeek
0,0,2024-03-23,12:29:38,CNR5884300,No Driver Found,CID1982111,eBike,Palam Vihar,Jhilmil,NaN,...,0,NaN,NaN,NaN,NaN,NaN,NaN,2024-03-23 12:29:38,12,Saturday
1,1,2024-11-29,18:01:39,CNR1326809,Incomplete,CID4604802,Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,...,1,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI,2024-11-29 18:01:39,18,Friday
2,2,2024-08-23,08:56:10,CNR8494506,Completed,CID9202816,Auto,Khandsa,Malviya Nagar,13.4,...,0,NaN,627.0,13.58,4.9,4.9,Debit Card,2024-08-23 08:56:10,8,Friday
3,3,2024-10-21,17:17:25,CNR8906825,Completed,CID2610914,Premier Sedan,Central Secretariat,Inderlok,13.1,...,0,NaN,416.0,34.02,4.6,5.0,UPI,2024-10-21 17:17:25,17,Monday
4,4,2024-09-16,22:08:00,CNR1950162,Completed,CID9933542,Bike,Ghitorni Village,Khan Market,5.3,...,0,NaN,737.0,48.21,4.1,4.3,UPI,2024-09-16 22:08:00,22,Monday


In [ ]:
df.isnull().sum()

,0
Date,0
Time,0
Booking ID,0
Booking Status,0
Customer ID,0
Vehicle Type,0
Pickup Location,0
Drop Location,0
Avg VTAT,10500
Avg CTAT,48000


In [ ]:
df['Row_ID'].count()

np.int64(150000)

In [ ]:
df[df['Booking Status'] != 'Completed']['Booking Value'].notna().sum()

np.int64(9000)

In [ ]:
anomaly = df[(df['Booking Status'] != 'Completed') & (df['Booking Value'].notna())]
print(anomaly['Booking Status'].value_counts())
anomaly[['Booking Status','Booking Value','Ride Distance']].head(5)

Booking Status
Incomplete    9000
Name: count, dtype: int64


,Booking Status,Booking Value,Ride Distance
1,Incomplete,237.0,5.73
9,Incomplete,135.0,10.36
28,Incomplete,304.0,1.98
42,Incomplete,966.0,18.30
47,Incomplete,453.0,19.24


9,000 "Incomplete" rides with a Booking Value

"Incomplete" means the ride started (driver picked up customer) but didn't finish (e.g., vehicle breakdown), so a fare/distance was already accruing. Only "Cancelled by Customer," "Cancelled by Driver," and "No Driver Found" should have null Booking Value, since those rides never started. This confirms data is internally consistent ,no fix needed.

In [ ]:
df['Booking Status'].value_counts()

,count
Booking Status,
Completed,93000
Cancelled by Driver,27000
No Driver Found,10500
Cancelled by Customer,10500
Incomplete,9000


150,000 total bookings,
* only **93,000 (62%)** were successfully completed meaning **38%** of all demand failed to convert into a completed ride.
* The largest driver of failure is **driver-side cancellations (27,000 / 18%)**, more than double **customer-side cancellations (10,500 / 7%)**.
* Supply shortage is also material: **10,500 bookings (7%)** had no driver found at all.
* A further **9,000 rides (6%)** started but didn't finish, indicating in-trip operational failures (e.g., breakdowns) rather than booking-stage issues.
* **1,233 Booking ID** collisions were identified as a data-generation defect , row count, not Booking ID, should be used as the booking grain going forward.

In [ ]:
df.to_excel('uber_cleaned.xlsx', index=False)

In [ ]:
from google.colab import files

files.download('uber_cleaned.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>